## Robustness Checks 

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

project_root = Path.cwd()
sys.path.append(str(project_root / "code"))

from robustness_alternative_inference import build_panel, stars


# -----------------------------
# Prepare data
# -----------------------------

df = build_panel().copy()

df["price_ihs"] = np.arcsinh(df["price"])
df["log_population_centered"] = df["log_population"] - df["log_population"].mean()
df["log_population_centered_sq"] = df["log_population_centered"] ** 2


# -----------------------------
# Run functional-form models with HC3 SEs
# -----------------------------

formulas = {
    "Baseline": (
        "log_price ~ treat_post + log_population + C(suburb) + C(year)"
    ),
    "Levels model": (
        "price ~ treat_post + log_population + C(suburb) + C(year)"
    ),
    "Quadratic population control": (
        "log_price ~ treat_post + log_population_centered "
        "+ log_population_centered_sq + C(suburb) + C(year)"
    ),
    "IHS outcome": (
        "price_ihs ~ treat_post + log_population + C(suburb) + C(year)"
    ),
    "Interaction added": (
        "log_price ~ treat_post + log_population_centered "
        "+ treat_post:log_population_centered + C(suburb) + C(year)"
    ),
}

results = {
    label: smf.ols(formula, data=df).fit(cov_type="HC3")
    for label, formula in formulas.items()
}


# -----------------------------
# Run 2017 pretrend test with HC3 SEs
# -----------------------------

panel_df = build_panel().copy()
pre_years_2017 = [2015, 2016]

for year in pre_years_2017:
    panel_df[f"treated_x_{year}"] = (
        panel_df["treated"] * (panel_df["year"] == year).astype(int)
    )

pretrend_2017_formula = (
    "log_price ~ "
    + " + ".join(f"treated_x_{year}" for year in pre_years_2017)
    + " + log_population + C(suburb) + C(year)"
)

pretrend_2017_result = smf.ols(
    pretrend_2017_formula,
    data=panel_df
).fit(cov_type="HC3")


# -----------------------------
# Helpers
# -----------------------------

def coefficient(result, variable):
    return f"{result.params[variable]:.3f}{stars(result.pvalues[variable])}"


def standard_error(result, variable):
    return f"({result.bse[variable]:.3f})"


# -----------------------------
# Robustness table
# -----------------------------

wald_2017 = pretrend_2017_result.wald_test(
    " = 0, ".join(f"treated_x_{year}" for year in pre_years_2017) + " = 0",
    scalar=True,
)

robustness_table = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Coefficient": coefficient(results["Baseline"], "treat_post"),
            "Standard Error": standard_error(results["Baseline"], "treat_post"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(results['Baseline'].nobs):,}",
            "R²": f"{results['Baseline'].rsquared:.3f}",
        },
        {
            "Model": "Levels model",
            "Coefficient": coefficient(results["Levels model"], "treat_post"),
            "Standard Error": standard_error(results["Levels model"], "treat_post"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(results['Levels model'].nobs):,}",
            "R²": f"{results['Levels model'].rsquared:.3f}",
        },
        {
            "Model": "Quadratic population control",
            "Coefficient": coefficient(results["Quadratic population control"], "treat_post"),
            "Standard Error": standard_error(results["Quadratic population control"], "treat_post"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(results['Quadratic population control'].nobs):,}",
            "R²": f"{results['Quadratic population control'].rsquared:.3f}",
        },
        {
            "Model": "IHS outcome",
            "Coefficient": coefficient(results["IHS outcome"], "treat_post"),
            "Standard Error": standard_error(results["IHS outcome"], "treat_post"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(results['IHS outcome'].nobs):,}",
            "R²": f"{results['IHS outcome'].rsquared:.3f}",
        },
        {
            "Model": "Interaction added",
            "Coefficient": coefficient(results["Interaction added"], "treat_post"),
            "Standard Error": standard_error(results["Interaction added"], "treat_post"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(results['Interaction added'].nobs):,}",
            "R²": f"{results['Interaction added'].rsquared:.3f}",
        },
        {
            "Model": "Pretrend test: Treat x 2015",
            "Coefficient": coefficient(pretrend_2017_result, "treated_x_2015"),
            "Standard Error": standard_error(pretrend_2017_result, "treated_x_2015"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(pretrend_2017_result.nobs):,}",
            "R²": f"{pretrend_2017_result.rsquared:.3f}",
        },
        {
            "Model": "Pretrend test: Treat x 2016",
            "Coefficient": coefficient(pretrend_2017_result, "treated_x_2016"),
            "Standard Error": standard_error(pretrend_2017_result, "treated_x_2016"),
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(pretrend_2017_result.nobs):,}",
            "R²": f"{pretrend_2017_result.rsquared:.3f}",
        },
        {
            "Model": "Joint pretrend test",
            "Coefficient": f"p = {float(wald_2017.pvalue):.3f}",
            "Standard Error": "",
            "Controls": "Yes",
            "Fixed Effects": "Yes",
            "SE Type": "HC3",
            "N": f"{int(pretrend_2017_result.nobs):,}",
            "R²": f"{pretrend_2017_result.rsquared:.3f}",
        },
    ]
)

display(robustness_table)


,Model,Coefficient,Standard Error,Controls,Fixed Effects,SE Type,N,R²
0,Baseline,-0.075***,(0.017),Yes,Yes,HC3,490,0.983
1,Levels model,-41464.104***,(13846.366),Yes,Yes,HC3,490,0.983
2,Quadratic population control,-0.072***,(0.017),Yes,Yes,HC3,490,0.983
3,IHS outcome,-0.075***,(0.017),Yes,Yes,HC3,490,0.983
4,Interaction added,-0.074***,(0.017),Yes,Yes,HC3,490,0.983
5,Pretrend test: Treat x 2015,0.007,(0.027),Yes,Yes,HC3,490,0.982
6,Pretrend test: Treat x 2016,0.024,(0.030),Yes,Yes,HC3,490,0.982
7,Joint pretrend test,p = 0.719,,Yes,Yes,HC3,490,0.982


### Interpretation

The robustness checks support the main DiD result. In the baseline specification, the estimated Treat × Post coefficient is negative, indicating that treated suburbs experienced lower house prices relative to control suburbs after the treatment period, conditional on population controls, suburb fixed effects, and year fixed effects.

The alternative functional-form checks show that this negative relationship is stable. The coefficient remains negative when the dependent variable is measured in levels, when a quadratic population control is added, when the inverse hyperbolic sine transformation of price is used, and when the model allows the treatment effect to vary with population size. This suggests that the main finding is not driven by one particular functional-form choice.

The pretrend test provides an additional check on the DiD identification assumption. The treated-by-year coefficients for 2015 and 2016 are small and statistically insignificant. The joint pretrend test also has a high p-value, meaning the null hypothesis that the pre-treatment coefficients are jointly equal to zero is not rejected. This provides no evidence that treated and control suburbs were already following different trends before the 2017 milestone.

Overall, the robustness checks strengthen confidence in the main result. The estimated effect remains negative across alternative model specifications, and the pretrend test does not indicate a violation of parallel trends before the milestone year. Standard errors are calculated using HC3 heteroskedasticity-robust standard errors, which make the inference less sensitive to unequal error variance across observations.
